In [ ]:
# 실습에 필요한 패키지 설치 (최초 1회 실행)
!pip install -q openai python-dotenv httpx

# 1. 환경 설정 및 핵심 라이브러리

LLM API를 활용한 데이터 생성 및 증강 작업을 진행할 예정이다.
API 키를 안전하게 관리하고, 효율적인 통신을 위한 기본 설정을 먼저 구성한다.

> **📌 이 노트북에서 익힐 것**
>
> | 챕터 | 내용 | 왜 필요한가 |
> |---|---|---|
> | **1-1** | 환경 변수와 `.env` | API 키를 안전하게 다루기 |
> | **1-2** | GMS 연결 + Responses API | LLM을 실제로 호출하기 |
> | **1-3** | JSON 구조화 출력 | 응답을 정해진 형식으로 받기 |
> | **1-4** | 비동기 통신 | 여러 요청을 동시에 보내기 |
>
> 네 가지 모두 다음 노트북(`2_2_Synthesis_data`)에서 그대로 쓰인다.

## 1-1. 환경 변수

### 1-1-1. 환경 변수란?

- 운영체제가 프로그램 실행 환경에 제공하는 **전역 설정값**
- 프로그램은 이 변수를 통해 API 키, DB 비밀번호 등 민감한 정보를 **코드 외부에서** 읽어올 수 있다

### 1-1-2. 환경 변수 설정 이유

1. **보안**: API KEY를 코드에 직접 작성하면 GitHub 등에 공유 시 유출 위험이 있다. 환경 변수를 통해 민감 정보를 코드와 분리하여 관리한다.
2. **유연성**: 개발/테스트/배포 환경마다 다른 API KEY를 사용할 때, 코드를 수정하지 않고 환경 변수 값만 변경하면 된다.

> **⚠️ 키 유출은 실제로 매일 일어나는 사고다**
>
> GitHub에 올라온 API 키를 자동으로 수집하는 봇이 상시 돌아다닌다.
> 커밋 후 몇 분 만에 발견되어 남의 키로 API가 호출되고, 요금이 청구되는 사례가 계속 나온다.
>
> 더 무서운 것은 **한 번 커밋하면 지워도 이력에 남는다**는 점이다.
> 파일을 삭제하고 다시 커밋해도 이전 커밋에는 키가 그대로 있다.
>
> 👉 **키가 유출되었다면 지우려 하지 말고, 즉시 발급처에서 키를 폐기하고 새로 발급받아야 한다.**

### 1-1-3. .env 파일 생성 방법

먼저 [GMS 사이트](https://gms.ssafy.io/web/)에 접속하여 본인의 **API Key를 복사**한다.

그리고 이 노트북과 **같은 폴더**에 `.env` 파일을 생성하고, 아래 내용을 작성한다.
따옴표 안에 복사한 키를 붙여넣으면 된다.

```
GMS_KEY="여기에_복사한_API키를_붙여넣기"
```

> **💡 .env 파일이란?**
>
> 키=값 형태로 환경 변수를 저장하는 텍스트 파일이다.
> `python-dotenv` 라이브러리가 이 파일을 읽어 운영체제의 환경 변수로 등록해 준다.
> `.gitignore`에 `.env`를 추가하면 GitHub에 실수로 올리는 것을 방지할 수 있다.

### 1-1-4. 실무에서는 이렇게 관리한다

**① `.gitignore`에 등록하기**

프로젝트 루트에 `.gitignore` 파일을 만들고 아래 한 줄을 넣는다.

```
.env
```

이러면 Git이 `.env`를 아예 추적하지 않아서, 실수로 커밋할 수가 없다.

**② 팀원에게는 `.env.example`을 공유한다**

`.env` 자체는 **절대 공유하지 않는다.** 대신 값이 빈 예시 파일을 만들어 함께 커밋한다.

```
# .env.example  (이 파일은 커밋해도 안전하다)
GMS_KEY=""
```

새로 합류한 팀원은 이 파일을 복사해 `.env`로 이름을 바꾸고, 자기 키를 채워 넣으면 된다.
**"어떤 키가 필요한지"는 공유하되, "값"은 공유하지 않는 것**이 핵심이다.

> **⚠️ 자주 발생하는 오류 — 실행 전에 확인하자**
>
> | 증상 | 원인 | 해결 |
> |---|---|---|
> | `GMS_KEY가 설정되지 않았습니다` | `.env`가 다른 폴더에 있음 | 노트북과 **같은 폴더**로 이동 |
> | 같은 증상 | 파일명이 `.env.txt` | 윈도우에서 확장자 표시를 켜고 이름 수정 |
> | 같은 증상 | 키 이름 오타 (`GMS_API_KEY` 등) | 정확히 `GMS_KEY` |
> | 같은 증상 | `.env`를 고친 뒤 그대로 실행 | **커널 재시작** 후 처음부터 재실행 |
>
> 마지막 항목이 특히 헷갈린다. `load_dotenv()`는 **셀을 실행하는 시점**에 파일을 읽으므로,
> 파일을 수정했다면 그 셀을 다시 실행해야 반영된다.

> **💡 Colab을 쓴다면**
>
> Colab에는 `.env` 대신 쓸 수 있는 **보안 비밀(Secrets)** 기능이 있다.
> 왼쪽 사이드바의 🔑 아이콘에서 이름과 값을 등록하고 아래처럼 꺼내 쓴다.
>
> ```python
> from google.colab import userdata
> GMS_KEY = userdata.get('GMS_KEY')
> ```
>
> 노트북을 공유해도 키는 함께 넘어가지 않아 안전하다.


In [ ]:
from dotenv import load_dotenv
from os import getenv

# .env 파일에서 환경 변수를 로드
# 왜 load_dotenv()를 호출하는가?
# → .env 파일에 적힌 키=값 쌍을 운영체제의 환경 변수로 등록한다.
#   이후 getenv()로 해당 값을 파이썬 코드 안에서 꺼내 쓸 수 있다.
load_dotenv()  # 현재 디렉토리의 .env 파일을 자동으로 찾아 로드

# 등록된 환경 변수에서 API 키를 가져온다
GMS_KEY = getenv('GMS_KEY')

if GMS_KEY:
    print('API 키 로드 성공!')
else:
    print('ERROR: .env 파일에 GMS_KEY가 설정되지 않았습니다.')
    print('이 노트북과 같은 폴더에 .env 파일을 생성하고 API 키를 입력하세요.')


## 1-2. GMS API 기본 사용 방법

### 1-2-1. openai 라이브러리 사용

SSAFY GMS는 OpenAI API를 중계(proxy)하는 서비스로, 요청/응답 구조가 OpenAI API와 **동일**하다.
따라서 `openai` 라이브러리를 그대로 사용하되, 요청을 보내는 서버 주소(`base_url`)만 GMS 엔드포인트로 변경하면 된다.

```
   내 코드  ──요청──>  GMS 서버  ──중계──>  OpenAI 서버
           <──응답──            <──응답──
```

> **💡 왜 openai 라이브러리를 쓰는가?**
>
> OpenAI가 만든 API 통신 규격이 업계 표준이 되었다.
> GMS는 물론이고 Azure OpenAI 등 다른 서비스도 이 규격을 지원하므로,
> `openai` 라이브러리 하나로 여러 회사의 LLM을 사용할 수 있다.
> 바꿔야 할 것은 `base_url`과 `api_key`뿐이다.

> **💡 API 호출에 꼭 필요한 세 가지**
>
> | 요소 | 역할 | 우리 코드에서는 |
> |---|---|---|
> | **주소(base_url)** | 어디로 보낼 것인가 | `https://gms.ssafy.io/gmsapi/...` |
> | **인증(api_key)** | 너는 누구인가 | `.env`에서 읽은 `GMS_KEY` |
> | **요청 내용** | 무엇을 해달라는 것인가 | `model`, `input` 등 |
>
> 이 세 가지는 어떤 API를 쓰든 똑같다. GMS든 날씨 API든 구조는 같다.

### 1-2-2. Responses API

OpenAI는 두 가지 방식의 API를 제공한다.

| 구분 | Chat Completions API | **Responses API** |
|------|---------------------|-------------------|
| 메서드 | `client.chat.completions.create()` | `client.responses.create()` |
| 입력 | `messages=[{'role':..., 'content':...}]` | `input='...'` (문자열도 가능) |
| 시스템 지시 | `messages`에 `role: 'system'` 추가 | `instructions='...'` 파라미터 |
| 결과 꺼내기 | `response.choices[0].message.content` | **`response.output_text`** |
| 위상 | 기존 방식 (유지보수 모드) | **권장 방식** |

**Responses API를 쓰는 이유**

- 입력과 출력 구조가 단순해졌다. 결과를 꺼낼 때 `choices[0].message.content` 같은 긴 경로가 필요 없다.
- 추론(reasoning) 과정, 도구 사용(tool use) 등 최신 기능이 이쪽에만 추가되고 있다.
- 이 실습에서는 처음부터 Responses API로 작성한다.

> **⚠️ 인터넷 예제는 대부분 옛날 방식이다**
>
> 검색으로 찾은 코드가 `chat.completions.create()`로 되어 있다면 **옛 방식**이다.
> 그대로 복붙하면 동작은 하지만, 이 노트북에서 배우는 것과 구조가 달라 헷갈린다.
> 위 표를 보고 바꿔서 쓰자.

> **💡 사용 가능한 모델**
>
> 이번 실습에서는 `gpt-5-nano` 모델을 사용한다.
> 교육 계정에서 사용할 수 있는 모델이 제한되어 있으므로, 모델명을 임의로 바꾸면 오류가 발생할 수 있다.

### 1-2-3. 토큰과 비용

LLM API는 **토큰(token) 단위로 과금**된다. 토큰은 모델이 글자를 처리하는 최소 단위다.

| 토큰 종류 | 무엇인가 | 과금 |
|---|---|---|
| **입력 토큰** | 우리가 보낸 프롬프트 | 있음 |
| **추론 토큰** | 모델이 답하기 전 속으로 생각한 분량 | **있음** (눈에 안 보임) |
| **출력 토큰** | 화면에 표시된 답변 | 있음 |

<br>

> **⚠️ 눈에 보이지 않는 것에도 돈이 나간다**
>
> 답변이 세 줄이라고 세 줄 값만 내는 것이 아니다.
> 추론 모델은 **속으로 생각한 분량(추론 토큰)** 도 함께 과금된다.
> 그래서 1-2-4에서 배울 `effort` 설정이 곧 비용 관리가 된다.

> **💡 토큰 감각 익히기**
>
> - 영어는 대략 **1토큰 ≈ 4글자** 정도로 쪼개진다
> - 한국어는 더 잘게 쪼개지는 편이라, 같은 의미의 문장도 토큰 수가 더 많이 나온다
> - 프롬프트를 길게 쓸수록(예: Few-shot 예시를 많이 넣을수록) 입력 토큰이 늘어난다
>
> 실무에서 "프롬프트를 짧게 유지하라"고 하는 이유가 여기에 있다.

### 1-2-4. 에러가 나면 — 상태 코드 읽는 법

API 호출이 실패하면 **HTTP 상태 코드**가 함께 온다. 코드만 봐도 원인을 좁힐 수 있다.

| 코드 | 의미 | 흔한 원인 | 대응 |
|:---:|---|---|---|
| **401** | 인증 실패 | 키가 틀렸거나 만료됨 | `.env` 확인, 키 재발급 |
| **400** | 잘못된 요청 | 모델명 오타, 지원하지 않는 파라미터, 스키마 오류 | 에러 메시지 본문을 읽을 것 |
| **404** | 대상 없음 | `base_url` 또는 경로가 틀림 | 주소 확인 |
| **429** | 요청이 너무 많음 | 짧은 시간에 과도한 호출 (Rate Limit) | 잠시 후 재시도 |
| **500 / 503** | 서버 오류 | 서비스 측 문제 | 잠시 후 재시도 |

<br>

> **💡 에러 메시지는 끝까지 읽자**
>
> 파이썬 에러는 길어서 겁이 나지만, **가장 중요한 정보는 맨 아래 한 줄**에 있다.
> 예를 들어 `Unsupported parameter: 'temperature' is not supported with this model.`
> 이라면 원인이 그대로 적혀 있는 것이다.
>
> 400 에러는 특히 **본문에 이유가 구체적으로 적혀 있으므로** 반드시 확인하자.


In [ ]:
from openai import OpenAI

# ========== GMS API 클라이언트 생성 ==========
# OpenAI 클래스에 GMS의 주소와 키를 넣으면 GMS를 통해 LLM과 통신할 수 있다.
client = OpenAI(
    api_key=GMS_KEY,                                            # .env에서 읽어온 API 키
    base_url='https://gms.ssafy.io/gmsapi/api.openai.com/v1/',  # GMS 서버 주소
)

# ========== 가장 단순한 호출 ==========
# client.responses.create: Responses API로 요청을 보내는 메서드
# input에는 문자열을 그대로 넣을 수 있다.
response = client.responses.create(
    model='gpt-5-nano',
    input='안녕? 넌 이름이 뭐니?',
)

# output_text: 모델이 생성한 최종 텍스트를 바로 꺼내주는 속성
print(response.output_text)


### 1-2-5. 스트리밍 응답

응답을 한 번에 받지 않고, 생성되는 대로 조각(chunk)씩 실시간으로 받아볼 수 있다.
사용자가 타이핑하듯 글자가 하나씩 나타나는 효과를 만들 때 사용한다.

> **💡 Responses API의 스트리밍은 '이벤트' 단위다**
>
> Chat Completions에서는 조각마다 `chunk.choices[0].delta.content`를 확인했다.
> Responses API는 **어떤 종류의 이벤트인지**(`event.type`)를 먼저 보고,
> 텍스트 조각 이벤트(`response.output_text.delta`)일 때만 `event.delta`를 꺼낸다.
>
> 이벤트 종류가 나뉘어 있어서, 나중에 도구 호출이나 추론 과정 같은
> 다른 이벤트도 같은 방식으로 처리할 수 있다.

> **💡 스트리밍은 '빨라지는 것'이 아니라 '빨라 보이는 것'이다**
>
> 전체 응답을 받는 데 걸리는 시간은 똑같다. 달라지는 것은 **체감**이다.
>
> | | 스트리밍 없음 | 스트리밍 |
> |---|---|---|
> | 첫 글자가 보이기까지 | 10초 (다 만들어질 때까지 대기) | 1초 |
> | 전체 완료까지 | 10초 | 10초 |
>
> ChatGPT가 글자를 하나씩 뿌리는 이유가 이것이다. **UX 기법**이다.

> **⚠️ 스트리밍이 어울리지 않는 경우**
>
> | 상황 | 이유 |
> |---|---|
> | **JSON 구조화 출력** | 조각난 JSON은 파싱할 수 없다. 다 모아야 의미가 생긴다 |
> | 배치 처리 | 사람이 보고 있지 않으므로 실시간일 필요가 없다 |
> | 결과를 후처리해야 할 때 | 어차피 전체가 필요하다 |
>
> 👉 **사람이 실시간으로 읽는 화면에만 쓴다.**
> 다음 노트북에서 데이터를 생성할 때 스트리밍을 쓰지 않는 이유가 이것이다.


In [ ]:
# ========== 스트리밍으로 응답 받기 ==========
stream = client.responses.create(
    model='gpt-5-nano',
    input='자기소개를 세 문장으로 해 줘.',
    stream=True,   # 응답을 조각 단위로 실시간 수신
)

for event in stream:
    # 텍스트 조각이 도착한 이벤트만 골라서 출력
    if event.type == 'response.output_text.delta':
        print(event.delta, end='')   # end='' -> 줄바꿈 없이 이어서 출력

print()   # 마지막 줄바꿈


### 1-2-4. 추론 강도 조절 (reasoning effort)

이전 세대 모델에서는 `temperature`나 `top_p`로 응답의 성향을 조절했다.
하지만 **GPT-5 계열의 추론(reasoning) 모델은 이 값들을 기본값 그대로 사용**하며,
다른 값을 지정하면 오류가 발생한다.

```
Unsupported parameter: 'temperature' is not supported with this model.
```

대신 **`reasoning`의 `effort`** 로 "얼마나 깊게 생각할지"를 조절한다.

| effort | 특징 | 적합한 작업 |
|--------|------|------------|
| `low` | 빠르고 저렴. 생각을 짧게 함 | 분류, 형식 변환, 단순 채점 |
| `medium` (기본값) | 균형 | 일반적인 생성 작업 |
| `high` | 느리고 비쌈. 오래 생각함 | 복잡한 추론, 수학, 코드 설계 |

<br>

> **💡 temperature와 무엇이 다른가?**
>
> - `temperature`: **답을 고를 때의 무작위성**을 조절 → 다양성이 달라짐
> - `effort`: **답을 내기 전 생각하는 양**을 조절 → 정확도와 비용이 달라짐
>
> 성격이 완전히 다른 손잡이다. effort를 높인다고 답변이 다양해지지는 않는다.

> **⚠️ effort가 높을수록 요금과 시간이 늘어난다**
>
> 눈에 보이지 않는 "생각하는 토큰"도 과금 대상이다.
> 무조건 `high`로 두지 말고, **작업 난이도에 맞춰** 고르는 습관을 들이자.


In [ ]:
import time

# ========== effort에 따른 차이 관찰 ==========
question = '한 변의 길이가 3인 정육면체의 겉넓이와 부피를 구하고, 그 비율을 설명해 줘.'

for effort in ['low', 'high']:
    start = time.time()
    response = client.responses.create(
        model='gpt-5-nano',
        input=question,
        reasoning={'effort': effort},   # 추론 강도 지정
    )
    elapsed = time.time() - start

    print('=' * 60)
    print(f'effort = {effort}  |  소요 시간 {elapsed:.1f}초')
    print('=' * 60)
    print(response.output_text)
    print()

# [관찰 포인트]
#  - high 쪽이 대체로 더 느리다. 답을 내기 전 더 오래 '생각'하기 때문이다.
#  - 이번 문제는 단순한 편이라 결과 차이가 크지 않을 수 있다.
#    복잡한 문제일수록 effort의 효과가 뚜렷해진다.
#  - 참고: Chat Completions API에서는 같은 기능을 reasoning_effort='high' 로 지정한다.


## 1-3. JSON (JavaScript Object Notation)

- 데이터를 저장하거나 주고받을 때 사용하는 **가볍고 사람이 읽기 쉬운** 데이터 형식
- 파이썬의 딕셔너리와 유사한 `"키": 값` 형태로 구성된다
- 대부분의 프로그래밍 언어와 API 통신에서 **표준**처럼 사용된다

### 1-3-1. 파이썬 딕셔너리와 비슷하지만 다르다

생김새가 거의 같아서 헷갈리기 쉽지만, JSON은 **문자열**이고 딕셔너리는 **파이썬 객체**다.

| | 파이썬 딕셔너리 | JSON |
|---|---|---|
| 정체 | 파이썬 자료구조 | **문자열** |
| 문자열 따옴표 | `'` 와 `"` 모두 가능 | **`"` 만 가능** |
| 참/거짓 | `True` / `False` | `true` / `false` |
| 빈 값 | `None` | `null` |
| 마지막 쉼표 | 허용 | **불허** |

**변환 함수 두 개만 기억하면 된다.**

```python
json.loads(문자열)   # JSON 문자열  ->  파이썬 딕셔너리   (load string)
json.dumps(딕셔너리)  # 파이썬 딕셔너리 -> JSON 문자열     (dump string)
```

> **⚠️ LLM 응답은 '딕셔너리처럼 보이는 문자열'이다**
>
> `response.output_text`로 받은 값은 아무리 딕셔너리처럼 생겼어도 **문자열**이다.
> `result['capital']` 처럼 바로 접근하면 오류가 난다.
> 반드시 `json.loads()`로 변환한 뒤에 사용해야 한다.

### 1-3-2. 왜 JSON을 강제하는가?

> LLM은 기본적으로 "자유로운 텍스트"로 응답한다.
> 하지만 합성 데이터를 만들 때는 **일관된 구조**가 필요하다.
> `text` 옵션에 JSON 스키마를 지정하면,
> LLM이 반드시 해당 구조에 맞춰 응답하도록 강제할 수 있다.

형식이 흔들리면 어떤 일이 생기는지 생각해 보자.
데이터 1,000건 중 950건은 `{"movie_name": ...}`인데 50건만 `{"title": ...}`이라면,
후속 코드가 오류로 멈추거나 **조용히 50건을 누락**시킨다. 후자가 훨씬 위험하다.

> **📌 Chat Completions와 옵션 이름이 다르다**
>
> | | Chat Completions | Responses |
> |---|---|---|
> | 옵션 이름 | `response_format=` | **`text=`** |
> | 스키마 위치 | `{'type':'json_schema', 'json_schema': {...}}` | `{'format': {'type':'json_schema', ...}}` |
>
> Responses API 쪽이 한 단계 덜 중첩되어 있다. 옛날 코드를 참고할 때 헷갈리기 쉬우니 주의하자.

### 1-3-3. strict 모드의 필수 조건

`'strict': True`를 쓰면 형식을 확실하게 보장받을 수 있다.
대신 스키마 작성 시 아래 조건을 **반드시** 지켜야 한다.

```
① properties의 모든 키가 required에 들어가야 한다
② additionalProperties 를 반드시 False 로 지정해야 한다
③ 스키마 name 에는 영문/숫자/언더바만 사용할 수 있다 (한글 불가)
```

하나라도 빠지면 **400 에러**가 발생한다. 외울 필요는 없고,
400 에러가 나면 이 세 가지부터 확인하면 된다.


In [ ]:
import json

# ========== JSON 응답 형식 정의 ==========
# LLM에게 "이 구조에 맞춰서 대답해"라고 강제하는 설정이다.
text_format = {
    'format': {
        'type': 'json_schema',       # 응답 형식을 JSON 스키마로 지정
        'name': 'capital_info',      # 스키마 이름 (영문/숫자/언더바만 허용)
        'strict': True,              # 엄격 모드: 스키마에 안 맞으면 에러
        'schema': {
            'type': 'object',        # 최상위 타입: 딕셔너리(객체)
            'properties': {          # 포함될 속성(키) 정의
                'capital': {'type': 'string'},
                'translation': {'type': 'string', 'description': '수도의 영어 번역'},
            },
            'required': ['capital', 'translation'],  # 반드시 포함해야 할 키
            # strict 모드에서는 아래 두 가지가 필수 조건이다.
            #   1) properties의 모든 키가 required에 들어가야 한다
            #   2) additionalProperties를 반드시 False로 지정해야 한다
            # 빠뜨리면 400 에러가 발생한다.
            'additionalProperties': False,
        },
    },
}

# ========== JSON 형식으로 응답 받기 ==========
response = client.responses.create(
    model='gpt-5-nano',
    input='한국의 수도는 어디야?',
    text=text_format,   # JSON 스키마 적용 (Chat Completions의 response_format에 해당)
)

# LLM 응답은 JSON 형식의 "문자열"이므로, 파이썬 딕셔너리로 변환해야 한다
# json.loads: JSON 문자열 → 파이썬 딕셔너리
structured_dictionary = json.loads(response.output_text)

print('구조화된 응답:')
for key, value in structured_dictionary.items():
    print(f'  {key}: {value}')


## 1-4. [참고] HTTPX (비동기 통신)

### 1-4-1. HTTPX란?

- Python용 HTTP 클라이언트 라이브러리
- 기존 `requests`와 사용법이 유사하면서 **비동기(Asynchronous)** 통신을 지원하는 것이 특징

> **💡 비동기 통신이란?**
>
> 여러 개의 API 요청을 보낼 때, 하나의 요청이 끝날 때까지 **기다리지 않고**
> 여러 요청을 **동시에 병렬로** 처리하는 방식이다.
>
> 예: 데이터 100건을 생성해야 할 때
> - 동기: 1건 요청 -> 응답 대기 -> 다음 1건 요청 -> ... (순차 처리, 느림)
> - 비동기: 100건 동시 요청 -> 결과를 모아서 처리 (병렬 처리, 빠름)
>
> ```
>    한 건에 3초 걸린다고 하면
>
>    순차 :  3초 x 100건 = 300초 (5분)
>    병렬 :  100건 동시 요청  -> 약 3~10초
> ```

> **💡 왜 이렇게까지 빨라지나?**
>
> API 요청 시간의 대부분은 **서버 응답을 기다리는 시간**이다.
> 내 컴퓨터가 일하는 것이 아니라 그냥 놀고 있는 것이다.
> 그 노는 시간에 다른 요청을 계속 보내면 되기 때문에 이런 차이가 난다.
>
> 반대로 말하면, **기다림이 없는 작업에는 비동기가 소용없다.**
> 이미지 처리나 대규모 계산처럼 CPU가 실제로 일하는 작업은 빨라지지 않는다.

### 1-4-2. 핵심 키워드

- `async def`: 이 함수가 **비동기 함수**임을 선언. 내부에서 `await` 사용 가능
- `await`: 비동기 작업이 완료될 때까지 기다리되, **다른 비동기 작업은 계속 실행**되도록 함
  - 즉, "기다리는 동안 다른 일 먼저 하고 있어!"라는 의미
- `asyncio.gather(*tasks)`: 여러 비동기 작업을 **동시에 실행**하고 모든 결과를 모아서 반환

> **💡 코루틴은 '주문서'다**
>
> `call_responses_api(...)` 처럼 비동기 함수를 호출해도 **바로 실행되지 않는다.**
> "실행 대기 상태"의 객체(코루틴)만 만들어질 뿐이다.
> `await`나 `asyncio.gather()`를 만나야 비로소 출발한다.
>
> 아래 코드의 출력 순서를 보면 이 점을 확인할 수 있다.
> "태스크 생성" 메시지가 **전부 먼저 찍힌 뒤에** 실제 호출이 시작된다.

### 1-4-3. 실무에서 주의할 점

> **⚠️ ① 무한정 동시에 보내면 안 된다 — Rate Limit**
>
> 대부분의 API는 "1분에 몇 회까지"라는 제한이 있다.
> 1,000건을 한 번에 던지면 **429 (Too Many Requests)** 오류로 대부분 실패한다.
>
> 실무에서는 동시 실행 개수를 제한한다. 예를 들어 한 번에 10건씩만 보내는 식이다.
> (`asyncio.Semaphore`를 사용하지만, 지금 단계에서는 개념만 알아두자)

> **⚠️ ② 하나가 실패하면 전부 날아간다**
>
> `asyncio.gather()`는 기본적으로 **작업 하나가 실패하면 전체가 예외로 종료**된다.
> 999건이 성공했어도 1건 때문에 결과를 모두 잃을 수 있다.
>
> 이를 막으려면 아래 옵션을 준다.
>
> ```python
> results = await asyncio.gather(*tasks, return_exceptions=True)
> ```
>
> 이러면 실패한 작업은 예외 객체로 결과 리스트에 담기고, 나머지는 정상적으로 반환된다.
> 대량 처리에서는 **거의 필수**로 붙이는 옵션이다.

> **⚠️ ③ 주피터 노트북에서만 되는 문법이 있다**
>
> 이 노트북의 마지막 셀에는 `await request(tasks)`가 함수 밖에 그냥 쓰여 있다.
> 원래 파이썬 스크립트(`.py`)에서는 **함수 밖에서 `await`를 쓸 수 없다.**
> 주피터 노트북이 내부적으로 이벤트 루프를 돌리고 있어서 가능한 것이다.
>
> `.py` 파일로 옮길 때는 아래처럼 감싸야 한다.
>
> ```python
> import asyncio
> asyncio.run(request(tasks))
> ```


In [ ]:
import httpx
import asyncio


# ========== Responses API 응답에서 텍스트만 꺼내는 함수 ==========
# 왜 이 함수가 필요한가?
# → openai 라이브러리를 쓰면 response.output_text 한 줄이면 되지만,
#   지금은 HTTP 요청을 직접 보내므로 원본 JSON을 직접 해석해야 한다.
#   Responses API의 output은 여러 항목의 리스트이고,
#   추론 모델은 앞쪽에 'reasoning' 항목이 먼저 오는 경우가 있다.
#   따라서 output[0]을 그냥 꺼내면 안 되고, 'message' 항목을 찾아야 한다.
def extract_output_text(data):
    """Responses API 원본 JSON에서 모델이 생성한 텍스트를 이어붙여 반환한다."""
    texts = []
    for item in data.get('output', []):
        if item.get('type') == 'message':
            for part in item.get('content', []):
                if part.get('type') == 'output_text':
                    texts.append(part.get('text', ''))
    return ''.join(texts)


# ========== 비동기 API 호출 함수 ==========
# 왜 별도의 함수로 분리하는가?
# → 여러 프롬프트를 동시에 요청하기 위해, 각 요청을 독립적인 "작업(Task)"으로 만들어야 한다.
#   이 함수 하나가 하나의 작업에 해당한다.
async def call_responses_api(url, headers, payload):
    print('    call_responses_api 시작')
    # httpx.AsyncClient: 비동기 HTTP 요청을 보내는 클라이언트
    # timeout=60.0: 60초 안에 응답이 없으면 에러 발생
    #   → 추론형 모델은 응답이 느릴 수 있어 넉넉하게 잡는다.
    #     ReadTimeout이 자주 발생하면 이 값을 더 늘려보자.
    async with httpx.AsyncClient(timeout=60.0) as client:
        # await: 응답이 올 때까지 비동기적으로 대기
        # → 이 요청이 기다리는 동안 다른 요청이 동시에 실행될 수 있다
        response = await client.post(url, headers=headers, json=payload)
        response.raise_for_status()  # HTTP 오류(4xx, 5xx) 발생 시 예외
        data = response.json()
        return extract_output_text(data)


In [ ]:
# ========== 여러 작업을 동시에 실행하는 메인 함수 ==========
async def request(tasks):
    print('모든 요청 동시 실행 시작')
    # asyncio.gather: 리스트의 모든 비동기 작업을 동시에 실행하고
    # 모든 작업이 완료될 때까지 기다린 후 결과를 리스트로 반환
    results = await asyncio.gather(*tasks)
    print('모든 요청 완료\n')

    for i, res in enumerate(results, 1):
        print(f'Response {i}')
        print(res)
        print('=' * 40)


# ========== API 설정 ==========
# openai 라이브러리를 쓰지 않고 HTTP 요청을 직접 보내는 방식이다.
# Responses API의 실제 경로는 /responses 이다. (기존 방식은 /chat/completions)
url = 'https://gms.ssafy.io/gmsapi/api.openai.com/v1/responses'
headers = {
    'Content-Type': 'application/json',
    # Bearer 인증: API 키를 'Bearer <키>' 형태로 헤더에 담아 보낸다.
    # openai 라이브러리는 이 작업을 내부에서 대신 해준다.
    'Authorization': f'Bearer {GMS_KEY}',
}

# ========== 동시에 보낼 프롬프트 준비 ==========
prompts = [
    '농담 하나만 해 줘',
    '프랑스의 수도는 어디야?',
]

# 각 프롬프트를 비동기 작업(Task)으로 변환
tasks = []
for prompt in prompts:
    payload = {
        'model': 'gpt-5-nano',
        'input': prompt,          # Responses API는 messages 대신 input을 쓴다
        'reasoning': {'effort': 'low'},   # 간단한 질문이므로 낮은 강도로 충분
        'stream': False,          # 비동기에서는 스트리밍 대신 한 번에 받는다
    }
    # 함수를 호출하지만 await가 없으므로 아직 실행되지 않는다
    # → "실행 대기 상태"의 코루틴 객체만 만들어진다
    print(f'태스크 생성: "{prompt}"')
    tasks.append(call_responses_api(url, headers, payload))
    print('  → 아직 API 호출 전 (대기 상태)')

# ========== 모든 태스크 동시 실행 ==========
# Jupyter Notebook에서는 await를 직접 사용할 수 있다
await request(tasks)


## 1-5. 마무리

### 이 노트북에서 익힌 것

| 챕터 | 핵심 | 다음 노트북에서 이렇게 쓰인다 |
|---|---|---|
| **1-1** 환경 변수 | 키는 `.env`에, 코드에는 이름만 | 동일하게 `GMS_KEY` 로드 |
| **1-2** Responses API | `responses.create()` + `output_text` | 호출 함수 `chat_completion()` 으로 감싸서 사용 |
| **1-2-4** effort | `temperature` 대신 생각의 양을 조절 | 생성은 `medium`, 채점은 `low` |
| **1-3** JSON | `text` 옵션으로 형식 강제 | 합성 데이터의 필드 구조 고정 |
| **1-4** 비동기 | 여러 요청을 동시에 | 대량 생성 시 활용 |

### 꼭 기억할 4가지

1. **키는 코드에 쓰지 않는다.** 유출되면 지우려 하지 말고 즉시 재발급받는다.
2. **`temperature`가 아니라 `effort`다.** 다양성이 아니라 생각의 양을 조절하는 손잡이다.
3. **형식은 부탁이 아니라 강제다.** JSON 스키마로 못 박아야 후속 처리가 자동화된다.
4. **에러 코드를 먼저 본다.** 401은 키, 400은 요청 내용, 429는 너무 많이 보낸 것이다.

### ➡️ 다음 노트북

`2_2_Synthesis_data.ipynb` 에서는 여기서 익힌 도구로
**학습 데이터를 직접 만들고, 그 품질을 LLM이 채점하는** 파이프라인을 구현한다.

```
   프롬프트 설계  ->  합성 데이터 생성  ->  LLM 채점  ->  선별
```

---

## [참고] 공식 문서

- [OpenAI Structured Model Outputs](https://platform.openai.com/docs/guides/structured-outputs?lang=python)
- [OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses)
- [Reasoning models](https://platform.openai.com/docs/guides/reasoning)
- [Error codes](https://platform.openai.com/docs/guides/error-codes)
